# 05 · Jev (TypeSafe System One) vs LLMs

**Jev** (TypeSafe AI, `jev-1.13`, early access since 2026-09) is not an LLM: it takes a *state* (text or JSON) plus typed questions and returns, in one non-autoregressive pass, a typed answer with **calibrated probabilities**:

| Primitive | Returns | Maps from task field |
|---|---|---|
| `Choice` | label + probabilities + confidence | `Literal[...]` |
| `Score` | expected value over an ordinal rubric | (not used yet) |
| `Noul` | P(yes) | `bool` |

Price: $0.042 per 1M input tokens, output free. Text only; English is its strongest language, so this notebook measures the **Spanish vs English gap**.

Questions:
1. How close is Jev to LLMs on accuracy / F1, per language?
2. Are its probabilities calibrated (ECE, reliability curve)?
3. **Confidence routing**: Jev answers when it is confident and escalates the rest to an LLM. What accuracy per dollar does that give?

Requires `TYPESAFE_API_KEY` in `.env` ([console](https://console.typesafe.ai/keys)) and the cached LLM results from notebook 04.

In [ ]:
%load_ext autoreload
%autoreload 2

import logging
import os

import pandas as pd
from dotenv import load_dotenv

from genaianalysis.utils.paths import data_processed_dir, project_dir

load_dotenv(project_dir(".env"))
os.environ.setdefault("PYDANTIC_AI_NO_BANNER", "1")
logging.basicConfig(level=logging.ERROR)  # WARNING shows every retry
pd.set_option("display.max_colwidth", 120)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from genaianalysis.backends.jev import run_jev
from genaianalysis.eval import score, with_gold
from genaianalysis.run import run_task
from genaianalysis.schema import load_jsonl
from genaianalysis.tasks.sentiment import MESSAGE_SENTIMENT, SENTIMENT_ARC

assert os.getenv("TYPESAFE_API_KEY"), "Set TYPESAFE_API_KEY in .env (early access: https://typesafe.ai)"
LLM = "google:gemini-flash-lite-latest"  # escalation model for routing; any model from notebook 04
N_TWEETS, N_CHATS, RPM = 100, 45, 12

In [ ]:
tweets = {lang: load_jsonl(data_processed_dir(f"tweets_{lang}_test_{N_TWEETS}.jsonl")) for lang in ["spanish", "english"]}
jev = {lang: await run_jev(MESSAGE_SENTIMENT, convs) for lang, convs in tweets.items()}
llm = {lang: await run_task(MESSAGE_SENTIMENT, convs, LLM, rpm=RPM) for lang, convs in tweets.items()}  # cache hits from 04

pd.concat([
    score(pd.concat([jev[lang], llm[lang]]), tweets[lang], "sentiment").assign(language=lang)
    for lang in tweets
])

## Calibration
If Jev says 0.8, is it right about 80% of the time? The diagonal is perfect calibration.

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4.5))
for lang, r in jev.items():
    d = with_gold(r.dropna(subset=["sentiment"]), tweets[lang], "sentiment")
    conf = d["sentiment_proba"].map(lambda p: max(p.values()))
    ok = d["gold"] == d["sentiment"]
    bins = pd.cut(conf, np.linspace(0, 1, 11))
    g = pd.DataFrame({"conf": conf, "ok": ok}).groupby(bins, observed=True).mean()
    ax.plot(g["conf"], g["ok"], "o-", label=lang)
ax.plot([0, 1], [0, 1], "--", color="gray")
ax.set(xlabel="predicted probability", ylabel="observed accuracy", title="Jev reliability")
ax.legend();

## Confidence routing
Jev answers when `confidence ≥ t`; everything else goes to the LLM. We sweep `t` and plot accuracy against the share of items escalated (a proxy for LLM cost and latency).

In [ ]:
def route(lang: str, t: float) -> dict:
    j = with_gold(jev[lang], tweets[lang], "sentiment").set_index("id")
    lr = llm[lang].set_index("id")
    use_jev = j["sentiment_confidence"].fillna(0) >= t
    pred = j["sentiment"].where(use_jev, lr["sentiment"].reindex(j.index))
    cost = j["cost_usd"].sum() + lr["cost_usd"].reindex(j.index)[~use_jev].sum()
    return {"t": t, "escalated": 1 - use_jev.mean(), "accuracy": (pred == j["gold"]).mean(), "cost_usd": cost}

curves = {lang: pd.DataFrame([route(lang, t) for t in np.linspace(0, 1, 21)]) for lang in tweets}
fig, ax = plt.subplots(figsize=(6, 4))
for lang, c in curves.items():
    ax.plot(c["escalated"], c["accuracy"], "o-", label=lang)
ax.set(xlabel="share escalated to LLM", ylabel="accuracy", title=f"Jev → {LLM} routing")
ax.legend();
curves["spanish"]

## Sentiment arc on synthetic chats
Jev answers only the `Literal` fields (initial / final sentiment). The free-text justifications stay with the LLM, which suggests a hybrid: Jev labels everything, the LLM explains only the cases worth reading.

In [ ]:
chats = load_jsonl(data_processed_dir(f"synthetic_es_{N_CHATS}.jsonl"))
jev_arc = await run_jev(SENTIMENT_ARC, chats)
llm_arc = await run_task(SENTIMENT_ARC, chats, LLM, rpm=RPM)
pd.concat([score(pd.concat([jev_arc, llm_arc]), chats, f).assign(field=f) for f in ["initial_sentiment", "final_sentiment"]])

## Findings
Record them in `docs/research.md`: per-language gap, ECE, and the routing threshold that matches LLM accuracy at the lowest escalation rate.